In [1]:
import numpy as np
import pandas as pd
from nltk.classify import NaiveBayesClassifier
from nltk.corpus import subjectivity
from nltk.sentiment import SentimentAnalyzer
from nltk.sentiment.util import *

import numpy as np 
import pandas as pd 
#pd.set_option('display.max_colwidth', -1)
from time import time
import re
import string
import os
from pprint import pprint
import collections

import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="darkgrid")
sns.set(font_scale=1.3)

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import classification_report

from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
#from sklearn.externals import joblib
import joblib as jb

#import gensim

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

np.random.seed(37)


In [2]:
# Reading the saved data pickle file
df  = pd.DataFrame()
df = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/pickled_filtered_lead_para_sp500_clean.pkl', compression=None)
df_texts = pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/texts.pkl', compression=None)

df_sentiment=pd.read_pickle('/Users/Farid Srouji/Documents/Research/master/financialmodelingprep/data/pkl/sentiment.pkl', compression=None)

In [3]:
df_texts

,Date,Open,High,Low,Close,Adj Close,Volume,0,1,2,...,14,15,16,17,18,19,20,21,22,23
Date,,,,,,,,,,,,,,,,,,,,,
2019-09-25,2019-09-25,17.049999,18.450001,15.690000,15.960000,15.960000,0,,,,...,HILTON EXPANSION BRINGS NEW BRANDS TO CANADA.H...,,,"CARNIVAL CORPORATION, GOVERNMENT OF THE BAHAMA...",,CONSTELLATION BRANDS ELECTS NEW BOARD MEMBERS....,SETS REGULAR QUARTERLY DIVIDEND OF $0.35PER SH...,,ABBOTT ANNOUNCED THE U.S. FOOD AND DRUG ADMINI...,
2019-09-26,2019-09-26,16.230000,17.090000,15.350000,16.070000,16.070000,0,,,,...,,HARVARD UNIVERSITY ESTABLISHES MRNA IMMUNOTHER...,,U.S. JURY RULES IN FAVOR OF EAGLE VIEW TECHNOL...,SETS QUARTERLY DIVIDEND OF $0.20PER SHARE.. UN...,,LYONDELLBASELL PRICES PUBLIC OFFERING OF GUARA...,,ABBVIE ANNOUNCED THAT THE U.S. FOOD AND DRUG A...,
2019-09-27,2019-09-27,15.770000,18.690001,15.150000,17.219999,17.219999,0,,,,...,INTERCONTINENTAL EXCHANGE PROVIDES STATEMENT A...,,BROWN-FORMAN CORPORATION APPOINTS SOPHIA ANGEL...,,,"COOPER COMPANIES INC - ON SEPT 27, CO ENTERED...","ORACLE SAYS IN APRIL, INVESTED $40 MILLION IN ...",,ABBOTT LABORATORIES ANNOUNCED NEW DATA THAT SU...,
2019-09-30,2019-09-30,17.230000,17.350000,16.200001,16.240000,16.240000,0,,,,...,LATE-BREAKING DATA SHOWS ABBOTT'S MITRACLIP™ I...,,KAREN WALKER JOINS INTEL AS SENIOR VICE PRESID...,,"LEGGETT & PLATT INC SAYS ON SEPT 25, MATTHEW ...",AMERICAN TOWER CORPORATION PRICES SENIOR NOTES...,FCB UNITES NORTH AMERICAN OFFICES UNDER NEW ST...,,"PARTECH, INC. ENTERED INTO AGREEMENT TO ACQUIR...",
2019-10-01,2019-10-01,16.020000,18.620001,15.790000,18.559999,18.559999,0,,,,...,JACK DANIEL - THE JACK DANIEL DISTILLERY INTRO...,,XILINX SAYS COLLABORATING WITH MICROSOFT ON AZ...,,DIGITAL REALTY ANNOUNCES PRICING OF PREFERRED ...,WESTERN DIGITAL CORP SAYS CEO STEPHEN MILLIGAN...,,,"ACCENTURE PLC PRESENTS AT ALL DAY DEVOPS, NOV-...",
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2021-08-02,2021-08-02,18.160000,19.870001,17.990000,19.459999,19.459999,0,,,,...,,"SYLMAR, CALIF.--(BUSINESS WIRE)--PPG (NYSE:PPG...","NEW YORK, AUG. 2, 2021 /PRNEWSWIRE/ -- WHY: R...","BOSTON, MASSACHUSETTS--(NEWSFILE CORP. - AUGUS...",PHOENIX--(BUSINESS WIRE)--FREEPORT-MCMORAN INC...,"STAMFORD, CONN., AUG. 2, 2021 /PRNEWSWIRE/ -- ...",SECURITIES LITIGATION PARTNER JAMES (JOSH) WIL...,,,
2021-08-03,2021-08-03,19.170000,20.440001,17.700001,18.040001,18.040001,0,,,,...,"NEW YORK, AUG. 3, 2021 /PRNEWSWIRE/ -- ATTORNE...",LOS ANGELES--(BUSINESS WIRE)---- $ATVI #ATVI--...,NEW YORK--(BUSINESS WIRE)--ASSURANT REPORTS SE...,"NEW YORK, AUG. 3, 2021 /PRNEWSWIRE/ -- POMERAN...","NEW YORK, NY / ACCESSWIRE / AUGUST 3, 2021 / T...","NEW YORK, AUG. 3, 2021 /PRNEWSWIRE/ -- ROSEN L...",,"NEW YORK, NY / ACCESSWIRE / AUGUST 3, 2021 / L...",BEIJING--(BUSINESS WIRE)--JUNIPER NETWORKS (NY...,"CHANDLER, ARIZ.--(BUSINESS WIRE)--IN THE RACE ..."
2021-08-04,2021-08-04,18.230000,18.900000,17.670000,17.969999,17.969999,0,,,,...,"RESTON, VA., AUG. 4, 2021 /PRNEWSWIRE/ -- THE ...",CHICAGO--(BUSINESS WIRE)--ADM'S (NYSE: ADM) BO...,"DANVERS, MASS.--(BUSINESS WIRE)--ABIOMED ANNOU...","NEW YORK, AUG. 4, 2021 /PRNEWSWIRE/ -- BERNSTE...","NEW YORK, NEW YORK--(NEWSFILE CORP. - AUGUST 4...",,,NEW YORK--(BUSINESS WIRE)--BRAGAR EAGEL & SQUI...,,


In [ ]:
df["articles"].describe()
df = df.join(df_sentiment)
df["vix"] = df["adj close"]#.apply(np.int)
df["vix_ln_return"]=np.log(df["vix"]/df["vix"].shift(1))
df['sentiment'] = df[["sentiment"]].apply(pd.to_numeric)
df["sentiment_ewm"]=df["sentiment"].ewm(10).mean()
 

In [ ]:
df["vix_ln_return"].plot(figsize=(10, 5),title='vix ln return')

In [ ]:
df_part1 = df.loc['2019-09-25':'2020-02-14']
df_part2 = df.loc['2020-02-15':'2020-05-31']
df_part3 = df.loc['2020-06-01':'2021-07-15']

In [ ]:
dfplot = df_part3[['vix_ln_return']].plot(figsize=(10, 5),title='VIX ln return 2020-06-01 to 2021-07-15')
fig = dfplot.get_figure()
fig.savefig("/Users/Farid Srouji/Documents/Research/VIXInterplaySentiment/master/financialmodelingprep/images/vix_ln_return.png")

fig.savefig("/Users/Farid Srouji/Documents/Research/VIXInterplaySentiment/master/financialmodelingprep/images/sentiment.png")
df_part3[['sentiment_ewm']].plot(figsize=(10, 5),title='Sentiment 2020-06-01 to 2021-07-15')
fig.savefig("/Users/Farid Srouji/Documents/Research/VIXInterplaySentiment/master/financialmodelingprep/images/sentiment_ewm.png")
dfplot = df[['sentiment']].plot(figsize=(10, 5),title='Sentiment 2020-06-01 to 2021-07-15')
fig = dfplot.get_figure()